# A.2: Few-shot prompting the base model

Turn the base model into a question answerer with examples in the prompt, and count what the examples cost.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Base and instruction-tuned models on the same requests
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

BASE, INSTRUCT = "HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-135M-Instruct"

tok = AutoTokenizer.from_pretrained(INSTRUCT)

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).eval()

inst = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=torch.float32).eval()

def generate(model, text, max_new_tokens=40):
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=False)

def chat(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

requests = ["What is the capital of France?",
            "Give me three tips for staying focused while studying.",
            "Translate 'good morning' into Spanish."]

for r in requests:
    print("REQUEST:", r)
    print("  base:    ", repr(generate(base, r)))
    print("  instruct:", repr(generate(inst, chat([{"role": "user", "content": r}]))))


In [ ]:
few_shot = ("Q: What is the capital of Italy?\nA: Rome.\n\n"
            "Q: What is the capital of Germany?\nA: Berlin.\n\n"
            "Q: What is the capital of Spain?\nA: Madrid.\n\n")
for q in ["What is the capital of France?", "What is the capital of Japan?"]:
    prompt = few_shot + f"Q: {q}\nA:"
    print(repr(q), "->", repr(generate(base, prompt, max_new_tokens=12)))
print("tokens in the examples:", len(tok(few_shot).input_ids),
      "| tokens in the question alone:", len(tok("Q: What is the capital of France?\nA:").input_ids))


**Expected output**

Output:

```text
'What is the capital of France?' -> ' Paris.\n\nQ: What is the capital of England'
'What is the capital of Japan?' -> ' Tokyo.\n\nQ: What is the capital of France'
tokens in the examples: 47 | tokens in the question alone: 12
```


In [ ]:
q = "Give me three tips for staying focused while studying."
print(repr(generate(base, few_shot + f"Q: {q}\nA:", max_new_tokens=40)))


**Expected output**

Output:

```text
' Read the book.\n\nQ: What is the capital of the United States?\nA: Washington.\n\nQ: What is the capital of the United Kingdom?\nA: London.'
```
